In [2]:
import numpy as np
import pandas as pd
import os, sys
import shutil
import datetime as dt

SEGMENT_PATH = 'output/Segmented_Data_24EI'
FEATURE_PATH = 'output/Feature_Data_24EI'


In [ ]:
data_list = []
for patient_id in filter(lambda x: os.path.isdir(os.path.join(SEGMENT_PATH, x)), os.listdir(SEGMENT_PATH)):
    data_path = os.path.join(SEGMENT_PATH, patient_id, "data")
    feature_path = os.path.join(FEATURE_PATH,patient_id, "data", "features")
    if not os.path.isdir(data_path):
        continue
    
    rr_folder = os.path.join(data_path, "rr")
    segments_folder = os.path.join(data_path, "segments")
    
    if os.path.isdir(rr_folder) and os.path.isdir(segments_folder):
        rr_files = [f for f in os.listdir(rr_folder) if f.endswith("_rr.txt")]
        
        for rr_file in rr_files:
            rr_path = os.path.join(rr_folder, rr_file)
            segment_file = rr_file.replace('_rr.txt', '.csv')
            segment_path = os.path.join(segments_folder, segment_file)
            feature_file = rr_file.replace('_rr.txt', '_features.csv')
            feature_path_content = os.path.join(feature_path,feature_file)
            data_list.append({
                "patient_id": patient_id,
                "rr_path": rr_path,
                "feature_path":feature_path_content,
                "segment_path": segment_path if os.path.exists(segment_path) else None
            })

# Create DataFrame

df = pd.DataFrame(data_list)
# # Display DataFrame
# import ace_tools as tools
# tools.display_dataframe_to_user(name="Patient Files Data", dataframe=df)
df.sort_values(by=["segment_path", "patient_id"])


In [10]:
segment = (pd.read_csv(df['segment_path'].iloc[0]))['PLETH'].values
# arr = segment['PLETH'].values

In [ ]:
import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
# import pywt
from scipy.fftpack import fft

def wavelength_to_frequency(wavelength_nm, scaling_factor=1e14):
    """
    Convert wavelength (nm) to a physiologically relevant frequency (Hz).
    """
    c = 3e8  # Speed of light in m/s
    freq_thz = c / (wavelength_nm * 1e-9)  # Convert nm to THz
    return freq_thz / scaling_factor  # Scale down to match PPG range

def preprocess_ppg(ppg_signal, fs=100, lowcut=0.5, highcut=10.0):
    """
    Bandpass filter the PPG signal.
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(2, [low, high], btype='band')
    filtered_signal = signal.filtfilt(b, a, ppg_signal)
    return filtered_signal

def compute_stft(ppg_signal, fs=100):
    """
    Compute Short-Time Fourier Transform (STFT) of the PPG signal.
    """
    f, t, Zxx = signal.stft(ppg_signal, fs, nperseg=512)
    return f, t, np.abs(Zxx)

def detect_elemental_presence(frequencies, power_spectrum, absorption_bands):
    """
    Check if any spectral peaks align with elemental frequencies.
    """
    detected_elements = {}
    for element, bands in absorption_bands.items():
        detected = []
        for band in bands:
            idx = (np.abs(frequencies - band)).argmin()
            if power_spectrum[idx] > np.percentile(power_spectrum, 95):
                detected.append(band)
        if detected:
            detected_elements[element] = detected
    return detected_elements

def plot_spectral_verification(ppg_data_csv=None, fs=100):
    """
    Load PPG CSV, analyze each segment, and detect elemental presence.
    """
    # df = pd.read_csv(ppg_data_csv)
    # segments = df.columns[1:]  # Assuming first column is time
    # segment = (pd.read_csv(df['segment_path'].iloc[0]))['PLETH'].values
    
    # Convert elemental wavelengths to frequency bands (Hz)
    absorption_bands = {
        'Fe': [wavelength_to_frequency(438), wavelength_to_frequency(527), wavelength_to_frequency(617)],
        'Zn': [wavelength_to_frequency(472), wavelength_to_frequency(481)],
        'Mg': [wavelength_to_frequency(518), wavelength_to_frequency(552)],
        'Ca': [wavelength_to_frequency(422), wavelength_to_frequency(445)]
    }
    
    detected_summary = {}
    # for segment in segments:
    # for i in np.arange(len(df)):
    for i in np.arange(0,2000):
        segment = (pd.read_csv(df['segment_path'].iloc[i]))['PLETH'].values
        ppg_signal = preprocess_ppg(segment, fs)
        # ppg_signal = segment
        
        f, t, Zxx = compute_stft(ppg_signal, fs)
        power_spectrum = np.mean(Zxx, axis=1)  # Average power across time
        
        detected = detect_elemental_presence(f, power_spectrum, absorption_bands)
        if detected != {}:
            detected_summary[i] = detected
            
            # plt.figure(figsize=(10, 6))
            # plt.pcolormesh(t, f, np.log(Zxx + 1e-6), shading='gouraud', cmap='jet')
            # plt.colorbar(label='Log Power')
            # plt.title(f'STFT Spectrogram - {i} - {segment}')
            # plt.xlabel('Time (s)')
            # plt.ylabel('Frequency (Hz)')
            
            # for element, bands in detected.items():
            #     for band in bands:
            #         plt.axhline(y=band, color='white', linestyle='--', label=f'{element} {band:.2e} Hz')
            
            # plt.legend()
            # plt.show()
    
    return detected_summary

# Example Usage:
# detected_summary = plot_spectral_verification("ppg_segments.csv")
# print(detected_summary)  # View which elements were detected in each 5-min segment
detected_summary = plot_spectral_verification()
print(detected_summary) 

## Harmonic Frequency Resonance

In [ ]:
import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
from scipy.fftpack import fft

def harmonic_scaling(f_tHz, n_min=14, n_max=42):
    """
    Compute biologically relevant harmonic subharmonics of elemental frequencies.
    """
    scaled_freqs = [f_tHz / (2**n) for n in range(n_min, n_max)]
    return [f for f in scaled_freqs if 0.5 <= f <= 5.0]  # Keep only PPG-relevant frequencies

def russell_harmonic_mapping():
    """
    Map Fe, Zn, Mg, Ca to speculative Hz based on Russell's octave-tone positions.
    """
    return {
        'Mg': [4.0],  # 6th octave, tone 2
        'Ca': [6.0],  # 7th octave, tone 2
        'Fe': [12.0], # 7th octave, tone 7
        'Zn': [18.0]  # 8th octave, tone 5
    }

def bio_validated_mapping():
    """
    Map elemental frequencies to known biological oscillations.
    """
    return {
        'Fe': [0.8, 1.2],  # Heme circulation
        'Zn': [1.5],       # Enzyme oscillation
        'Mg': [2.5],       # ATPase activity
        'Ca': [1.0, 3.5]   # Membrane depolarization
    }

def preprocess_ppg(ppg_signal, fs=100, lowcut=0.5, highcut=10.0):
    """
    Bandpass filter the PPG signal.
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(2, [low, high], btype='band')
    filtered_signal = signal.filtfilt(b, a, ppg_signal)
    return filtered_signal

def compute_stft(ppg_signal, fs=100):
    """
    Compute Short-Time Fourier Transform (STFT) of the PPG signal.
    """
    f, t, Zxx = signal.stft(ppg_signal, fs, nperseg=512)
    return f, t, np.abs(Zxx)

def detect_elemental_presence(frequencies, power_spectrum, absorption_bands):
    """
    Check if any spectral peaks align with elemental frequencies.
    """
    detected_elements = {}
    for element, bands in absorption_bands.items():
        detected = []
        for band in bands:
            idx = (np.abs(frequencies - band)).argmin()
            if power_spectrum[idx] > np.percentile(power_spectrum, 95):
                detected.append(band)
        if detected:
            detected_elements[element] = detected
    return detected_elements

def plot_spectral_verification(ppg_data_csv=None, fs=100):
    """
    Load PPG CSV, analyze each segment, and detect elemental presence.
    """
    # Harmonic scaling for elemental THz frequencies
    # absorption_bands = {
    #     'Fe': harmonic_scaling(486),
    #     'Zn': harmonic_scaling(623),
    #     'Mg': harmonic_scaling(543),
    #     'Ca': harmonic_scaling(674)
    # }

    # Harmonic scaling from spectral lines (Petty-inspired)
    absorption_bands = {
        'Fe': harmonic_scaling(617),  # ~486 nm = 617 THz
        'Zn': harmonic_scaling(1408), # ~213 nm = 1408 THz
        'Mg': harmonic_scaling(1052), # ~285 nm = 1052 THz
        'Ca': harmonic_scaling(710)   # ~422 nm = 710 THz
    }
    
    # Russell harmonic mapping
    russell_bands = russell_harmonic_mapping()
    for element in russell_bands:
        absorption_bands[element].extend(russell_bands[element])
    
    # Add biologically validated mappings
    # bio_mapping = bio_validated_mapping()
    # for element in bio_mapping:
    #     absorption_bands[element].extend(bio_mapping[element])
    
    detected_summary = {}
    for i in np.arange(0,10000):
        segment = (pd.read_csv(df['segment_path'].iloc[i]))['PLETH'].values
        ppg_signal = preprocess_ppg(segment, fs)
        f, t, Zxx = compute_stft(ppg_signal, fs)
        power_spectrum = np.mean(Zxx, axis=1)
        
        detected = detect_elemental_presence(f, power_spectrum, absorption_bands)
        if detected:
            detected_summary[i] = detected
            
            plt.figure(figsize=(10, 6))
            plt.pcolormesh(t, f, np.log(Zxx + 1e-6), shading='gouraud',)
            plt.colorbar(label='Log Power')
            plt.title(f'STFT Spectrogram - Segment {i}')
            plt.xlabel('Time (s)')
            plt.ylabel('Frequency (Hz)')
            
            for element, bands in detected.items():
                for band in bands:
                    plt.axhline(y=band, color='white', linestyle='--', label=f'{element} {band:.2e} Hz')
            
            plt.legend()
            plt.show()
    
    return detected_summary

# Example Usage:
detected_summary = plot_spectral_verification()
print(detected_summary)


## COmpile Data Continue

In [15]:
# List to store all feature data
feature_data_list = []

# Iterate through the DataFrame rows
for _, row in df.iterrows():
    feature_path = row["feature_path"]
    
    if not pd.isna(feature_path) and os.path.exists(feature_path):
        # Extract patient_id and timestamp from the file path
        patient_id = row["patient_id"]
        timestamp_str = os.path.basename(feature_path).split('_')[0]  # Extract '20220421T155446'
        timestamp = f"{timestamp_str[:4]}/{timestamp_str[4:6]}/{timestamp_str[6:8]}"

        # Load the feature file into a DataFrame
        feature_df = pd.read_csv(feature_path)

        # Add patient_id and timestamp columns
        feature_df.insert(0, "patient_id", patient_id)
        feature_df.insert(1, "timestamp", timestamp)

        # Append to list
        feature_data_list.append(feature_df)

# Concatenate all feature data into a single DataFrame
final_feature_df = pd.concat(feature_data_list, ignore_index=True) if feature_data_list else pd.DataFrame()


In [ ]:
final_feature_df.sort_values(by=["patient_id", "timestamp", "start"])


In [23]:
df_final = final_feature_df.copy()

# Remove 30s overlap by keeping only unique timestamps per date
if not df_final.empty and "start" in df_final.columns and "end" in df_final.columns:
    df_final["date"] = df_final["timestamp"]
    
    # Identify numeric columns for aggregation
    numeric_cols = df_final.select_dtypes(include=["number"]).columns.tolist()
    non_numeric_cols = ["patient_id", "date", "timestamp", "start", "end"]
    
    # Define aggregation methods
    aggregation_methods = {col: "mean" for col in numeric_cols if col not in non_numeric_cols}
    aggregation_methods.update({"start": "min", "end": "max"})
    
    # Perform groupby aggregation
    df_final = df_final.groupby(["patient_id", "date"], as_index=False).agg(aggregation_methods)

In [ ]:
df_final

In [26]:
df_final = final_feature_df.copy()
if not df_final.empty:
    df_final["date"] = df_final["timestamp"]
    for (patient_id, date), group_df in df_final.groupby(["patient_id", "date"]):
        filename = f"features_{patient_id}_{date.replace('/', '-')}.csv"
        group_df.to_csv(filename, index=False)
        print(f"Saved {filename}")